In [3]:
from datasets import load_dataset
from datasets.features import Value, Sequence
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def is_text_feature(feature):
    """
    Recursively detect whether a HF feature represents text.
    """
    if isinstance(feature, Value) and feature.dtype == "string":
        return True
    if isinstance(feature, Sequence):
        return is_text_feature(feature.feature)
    return False

def get_text_columns(ds):
    """
    Identify text-like columns in the training split.
    """
    text_cols = []
    for col, feature in ds["train"].features.items():
        if is_text_feature(feature):
            text_cols.append(col)
    return text_cols

def text_length_stats(ds, text_col):
    """
    Compute word-count statistics for a text column across splits.
    Handles strings and lists of strings.
    """
    lengths = []

    for split in ds:
        if text_col not in ds[split].features:
            continue

        for val in ds[split][text_col]:
            if isinstance(val, str):
                lengths.append(len(val.split()))
            elif isinstance(val, list):
                lengths.extend(len(v.split()) for v in val if isinstance(v, str))

    return pd.Series(lengths).describe()

def evaluate_all_configs(datasets_by_config):
    """
    Run schema-aware structural evaluation over all ChessQA configs.
    """
    results = {}

    for cfg, ds in datasets_by_config.items():
        text_cols = get_text_columns(ds)

        results[cfg] = {
            "num_train": len(ds["train"]),
            "text_columns": text_cols,
            "text_length_stats": {
                col: text_length_stats(ds, col)
                for col in text_cols
            }
        }

    return results


In [ ]:
CONFIGS = [
    "structural",
    "motifs",
    "short_tactics",
    "position_judgement",
    "semantic"
]

datasets_by_config = {
    cfg: load_dataset("wieeii/ChessQA-Benchmark", cfg)
    for cfg in CONFIGS
}

results = evaluate_all_configs(datasets_by_config)
results

In [4]:
for cfg, ds in datasets_by_config.items():
    print(f"\n=== {cfg.upper()} ===")
    for split in ds:
        print(f"  {split}: {list(ds[split].features.keys())}")

NameError: name 'datasets_by_config' is not defined

In [ ]:
import pandas as pd

def summarize_results(results):
    rows = []
    
    for cfg, info in results.items():
        row = {
            "config": cfg,
            "num_train": info["num_train"],
            # "text_columns": ", ".join(info["text_columns"])
        }
        
        # pick a few key text columns to summarize
        for col in ["question", "input", "correct_answer", "format_examples"]:
            if col in info["text_length_stats"]:
                stats = info["text_length_stats"][col]
                row[f"{col}_mean_len"] = stats["mean"]
                # row[f"{col}_max_len"] = stats["max"]
                # row[f"{col}_min_len"] = stats["min"]
            else:
                row[f"{col}_mean_len"] = None
                # row[f"{col}_max_len"] = None
                # row[f"{col}_min_len"] = None
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

summary_df = summarize_results(results)
summary_df


In [ ]:
# ===== Prepare short_tactics dataset for T5-small =====

from datasets import load_dataset, Dataset
import pandas as pd

# 1️⃣ Load short_tactics config
dataset = load_dataset("wieeii/ChessQA-Benchmark", "short_tactics")

# Inspect the columns
print(dataset["train"].column_names)

# 2️⃣ Function to combine input fields into single string for T5
def format_input(example):
    """
    Combine question + board input + optional format_examples into a single string
    """
    parts = []
    
    if "question" in example and example["question"]:
        parts.append(f"[QUESTION] {example['question']}")
        
    if "input" in example and example["input"]:
        # input may be FEN or other board description
        parts.append(f"[BOARD] {example['input']}")
        
    if "format_examples" in example and example["format_examples"]:
        # convert list of strings to a single string if needed
        fmt = example["format_examples"]
        if isinstance(fmt, list):
            fmt = " | ".join(fmt)
        parts.append(f"[EXAMPLES] {fmt}")
        
    return " ".join(parts)

# 3️⃣ Apply formatting to all splits
def prepare_dataset(ds):
    input_texts = []
    target_texts = []
    
    for example in ds:
        input_texts.append(format_input(example))
        # correct_answer is the target
        target_texts.append(example["correct_answer"])
    
    return Dataset.from_dict({
        "input_text": input_texts,
        "target_text": target_texts
    })

# 4️⃣ Prepare train / validation / test splits
train_ds = prepare_dataset(dataset["train"])
val_ds = prepare_dataset(dataset["validation"]) if "validation" in dataset else None
test_ds = prepare_dataset(dataset["test"]) if "test" in dataset else None

# 5️⃣ Quick preview
print("Train example:")
print(train_ds[0])


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# tokenize example
train_encodings = tokenizer(
    train_ds["input_text"], 
    truncation=True, 
    padding=True,
    max_length=128
)
train_labels = tokenizer(
    train_ds["target_text"], 
    truncation=True, 
    padding=True,
    max_length=32
)


In [5]:
# ===== Full T5-small Fine-tuning Pipeline for short_tactics =====

from datasets import load_dataset, Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

# ----------------------
# 1️⃣ Load and prepare dataset
# ----------------------
dataset = load_dataset("wieeii/ChessQA-Benchmark", "short_tactics")

def format_input(example):
    """Combine question, board, and optional examples into a single input string"""
    parts = []
    if example.get("question"):
        parts.append(f"[QUESTION] {example['question']}")
    if example.get("input"):
        parts.append(f"[BOARD] {example['input']}")
    if example.get("format_examples"):
        fmt = example["format_examples"]
        if isinstance(fmt, list):
            fmt = " | ".join(fmt)
        parts.append(f"[EXAMPLES] {fmt}")
    return " ".join(parts)

def prepare_dataset(ds):
    input_texts = []
    target_texts = []
    for example in ds:
        input_texts.append(format_input(example))
        target_texts.append(example["correct_answer"])
    return Dataset.from_dict({
        "input_text": input_texts,
        "target_text": target_texts
    })

train_ds = prepare_dataset(dataset["train"])
val_ds = prepare_dataset(dataset["validation"]) if "validation" in dataset else None
test_ds = prepare_dataset(dataset["test"]) if "test" in dataset else None

# ----------------------
# 2️⃣ Tokenizer and model
# ----------------------
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

max_input_length = 128
max_target_length = 32

# Tokenization function
def tokenize(batch):
    model_inputs = tokenizer(
        batch["input_text"], 
        truncation=True, 
        padding="max_length", 
        max_length=max_input_length
    )
    labels = tokenizer(
        batch["target_text"], 
        truncation=True, 
        padding="max_length", 
        max_length=max_target_length
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_encodings = train_ds.map(tokenize, batched=True)
if val_ds:
    val_encodings = val_ds.map(tokenize, batched=True)
else:
    val_encodings = None

# ----------------------
# 3️⃣ Data collator
# ----------------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ----------------------
# 4️⃣ Training arguments
# ----------------------
training_args = TrainingArguments(
    output_dir="./t5_short_tactics",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="steps" if val_encodings else "no",
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# ----------------------
# 5️⃣ Metrics function
# ----------------------
from datasets import load_metric
metric = load_metric("exact_match")  # compute exact SAN match

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return metric.compute(predictions=decoded_preds, references=decoded_labels)

# ----------------------
# 6️⃣ Trainer setup
# ----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encodings,
    eval_dataset=val_encodings,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ----------------------
# 7️⃣ Start training
# ----------------------
trainer.train()


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

C:\Users\ChrisCyr\OneDrive - personalmicrosoftsoftware.uci.edu\Documents\School\Year 5\Q2 - Winter\cs175\Project\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ChrisCyr\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# Evaluate on test set if available
if test_ds:
    test_encodings = test_ds.map(tokenize, batched=True)
    results = trainer.evaluate(test_encodings)
    print(results)


In [ ]:
TACTICAL_KEYWORDS = ["mate", "check", "fork", "pin", "skewer"]

def keyword_coverage(ds):
    counts = {k: 0 for k in TACTICAL_KEYWORDS}
    total = 0

    for split in ds:
        for q in ds[split]["question"]:
            q_lower = q.lower()
            total += 1
            for k in TACTICAL_KEYWORDS:
                if k in q_lower:
                    counts[k] += 1

    return {k: v / total for k, v in counts.items()}

keyword_coverage(dataset)


In [ ]:
def dataset_readiness(ds):
    return {def export_clean_subset(ds, n=1000, split="train"):
    df = ds[split].to_pandas()
    df = df.dropna()
    return df.sample(min(n, len(df)), random_state=42)

sample_df = export_clean_subset(dataset)
sample_df.head()
        "fen_valid": fen_validity_report(ds),
        "question_len": text_length_stats(ds, "question"),
        "answer_len": text_length_stats(ds, "answer"),
        "answer_diversity": answer_type_stats(ds)["unique_answers"]
    }

dataset_readiness(dataset)

In [ ]:
def export_clean_subset(ds, n=1000, split="train"):
    df = ds[split].to_pandas()
    df = df.dropna()
    return df.sample(min(n, len(df)), random_state=42)

sample_df = export_clean_subset(dataset)
sample_df.head()